# GEDI Canopy Height Pipeline

This notebook processes NASA GEDI Level 2A HDF5 files stored in S3 through four sequential phases:

| Phase | Description |
|---|---|
| **Phase 1** | High-performance batch extraction of GEDI shots from S3 |
| **Phase 1a** | Year parsing from GEDI filenames |
| **Phase 1b** | Harmonization to the SMAP 9 km grid |
| **Phase 1c** | Spatial join to assign shots to study-area jurisdictions |

**Source:** `s3://central-virginia-tree-canopy-project/GEDI/`  
**Final output:** `s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary.csv`

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

packages = ["s3fs", "h5py", "geopandas", "xarray", "netCDF4", "pyarrow"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All dependencies installed.")

## Cell 2 — Imports

In [ ]:
import os
import io
import re
import urllib.request

import boto3
import s3fs
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

print("Imports successful.")

## Cell 3 — Configuration

Edit this cell to change S3 paths, bounding box, grid resolution, or target jurisdictions.

In [ ]:
# ── S3 settings ───────────────────────────────────────────────────────────────
S3_BUCKET             = "central-virginia-tree-canopy-project"
S3_PREFIX             = "GEDI/"
GEDI_COUNTY_S3_PREFIX = "gedi-county-summary/"

# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR        = "/home/ec2-user/SageMaker/gedi_output"
OUTPUT_PARQUET    = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_multiyear.parquet")
OUTPUT_NETCDF     = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_grid.nc")
OUTPUT_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_summary.csv")

# ── Spatial bounds (Virginia 8-jurisdiction study area) ───────────────────────
MIN_LON, MIN_LAT, MAX_LON, MAX_LAT = -79.1721, 37.3296, -77.6873, 38.4755

# ── SMAP grid resolution (~9 km) ─────────────────────────────────────────────
GRID_RES = 0.081

# ── Target jurisdictions (Name, State FIPS, County/Place FIPS, Type) ─────────
TARGET_JURISDICTIONS = [
    ("Albemarle",       "51", "003",   "county"),
    ("Augusta",         "51", "015",   "county"),
    ("Charlottesville", "51", "14968", "place"),
    ("Fluvanna",        "51", "065",   "county"),
    ("Greene",          "51", "079",   "county"),
    ("Louisa",          "51", "109",   "county"),
    ("Nelson",          "51", "125",   "county"),
    ("Rockingham",      "51", "165",   "county"),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  GEDI source  : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  S3 output    : s3://{S3_BUCKET}/{GEDI_COUNTY_S3_PREFIX}")

## Cell 4 — Helper Functions

In [ ]:
def parse_year_from_filename(filename: str):
    """Extract the year from a standard GEDI filename (e.g., GEDI02_A_2022143...)."""
    year_match = re.search(r'GEDI02_A_(\d{4})', filename)
    if year_match:
        return int(year_match.group(1))
    return None


def fetch_boundary(name: str, state_fips: str, geo_id: str, geo_type: str) -> gpd.GeoDataFrame:
    """Fetch boundary GeoJSON directly from the US Census TIGERweb API."""
    if geo_type == "place":
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "Places_CouSub_ConCity_SubMCD/MapServer/4/query"
            f"?where=STATE='{state_fips}'+AND+PLACE='{geo_id}'"
            "&outFields=NAME,STATE,PLACE&f=geojson&outSR=4326"
        )
    else:
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "State_County/MapServer/11/query"
            f"?where=STATE='{state_fips}'+AND+COUNTY='{geo_id}'"
            "&outFields=NAME,STATE,COUNTY&f=geojson&outSR=4326"
        )
    print(f"  Fetching boundary for {name}...")
    with urllib.request.urlopen(url, timeout=20) as r:
        gdf = gpd.read_file(r)
    if gdf.empty:
        raise ValueError(f"No boundary found for {name} (FIPS {state_fips}/{geo_id})")
    gdf = gdf.set_crs("EPSG:4326")
    gdf['jurisdiction'] = name
    return gdf


print("Helper functions defined.")

## Cell 5 — Phase 1 & 1a: Discover GEDI Files in S3

In [ ]:
print(f"Scanning s3://{S3_BUCKET}/{S3_PREFIX} for GEDI HDF5 files...")

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

h5_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".h5"):
            h5_keys.append(obj["Key"])

print(f"Found {len(h5_keys)} GEDI HDF5 files.")
if h5_keys:
    print(f"  First : {h5_keys[0]}")
    print(f"  Last  : {h5_keys[-1]}")

## Cell 6 — Phase 1 & 1a: Batch Extraction with Spatial Masking and Quality Filtering

In [ ]:
fs = s3fs.S3FileSystem(anon=False)
all_dfs = []

for i, key in enumerate(h5_keys):
    file_name = os.path.basename(key)

    # Phase 1a: Parse year from filename
    year = parse_year_from_filename(file_name)
    if not year:
        print(f"Warning: Could not parse year from {file_name}. Skipping.")
        continue

    try:
        # Stream HDF5 file from S3 into memory
        s3_path = f"s3://{S3_BUCKET}/{key}"
        with fs.open(s3_path, "rb") as f:
            raw_bytes = f.read()

        with h5py.File(io.BytesIO(raw_bytes), 'r') as hf:
            beams = [k for k in hf.keys() if k.startswith('BEAM')]

            for beam in beams:
                if 'lat_lowestmode' not in hf[beam]:
                    continue

                # Step 1: Extract coordinates first for rapid spatial masking
                lats = hf[f"{beam}/lat_lowestmode"][:]
                lons = hf[f"{beam}/lon_lowestmode"][:]

                spatial_mask = (
                    (lons >= MIN_LON) & (lons <= MAX_LON) &
                    (lats >= MIN_LAT) & (lats <= MAX_LAT)
                )

                if not np.any(spatial_mask):
                    continue

                # Step 2: Extract attributes only for points inside the bbox
                quality    = hf[f"{beam}/quality_flag"][:][spatial_mask]
                sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
                rh98       = hf[f"{beam}/rh"][:, 98][spatial_mask]

                beam_df = pd.DataFrame({
                    'longitude':    lons[spatial_mask],
                    'latitude':     lats[spatial_mask],
                    'quality_flag': quality,
                    'sensitivity':  sensitivity,
                    'rh98':         rh98,
                    'year':         year,
                    'file_source':  file_name,
                    'beam':         beam
                })

                # Step 3: Strict quality filtering
                valid_df = beam_df[
                    (beam_df['quality_flag'] == 1) &
                    (beam_df['sensitivity']  > 0.9) &
                    (beam_df['rh98']         > 0)   &
                    (beam_df['rh98']         < 100)
                ]

                if not valid_df.empty:
                    all_dfs.append(valid_df)

    except Exception as e:
        print(f"Error reading {file_name}: {e}")

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(h5_keys)} files...")

print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")

## Cell 7 — Save Extracted Points to Parquet

In [ ]:
if not all_dfs:
    raise RuntimeError("No valid GEDI shots found within the bounding box. Check S3 prefix and bbox settings.")

df_gedi = pd.concat(all_dfs, ignore_index=True)
df_gedi.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Total valid GEDI shots saved : {len(df_gedi):,}")
print(f"Years covered                : {sorted(df_gedi['year'].unique())}")
print(f"Parquet file                 : {OUTPUT_PARQUET}")
df_gedi.head()

## Cell 8 — Phase 1b: Harmonize to the SMAP 9 km Grid

In [ ]:
lon_bins = np.arange(MIN_LON, MAX_LON, GRID_RES)
lat_bins = np.arange(MIN_LAT, MAX_LAT, GRID_RES)

df_gedi['lon_grid'] = pd.cut(df_gedi['longitude'], bins=lon_bins, labels=lon_bins[:-1]).astype(float)
df_gedi['lat_grid'] = pd.cut(df_gedi['latitude'],  bins=lat_bins, labels=lat_bins[:-1]).astype(float)

gedi_grid = df_gedi.groupby(['year', 'lat_grid', 'lon_grid'])['rh98'].mean().reset_index()

ds_gedi = gedi_grid.set_index(['year', 'lat_grid', 'lon_grid']).to_xarray()
ds_gedi.to_netcdf(OUTPUT_NETCDF)

print(f"Grid cells produced : {len(gedi_grid):,}")
print(f"NetCDF saved to     : {OUTPUT_NETCDF}")
gedi_grid.head()

## Cell 9 — Phase 1c: Fetch Jurisdiction Boundaries from US Census TIGERweb

In [ ]:
boundary_gdfs = []

for name, st_fips, geo_id, geo_type in TARGET_JURISDICTIONS:
    try:
        b_gdf = fetch_boundary(name, st_fips, geo_id, geo_type)
        boundary_gdfs.append(b_gdf)
    except Exception as e:
        print(f"  Failed to fetch boundary for {name}: {e}")

if not boundary_gdfs:
    raise RuntimeError("No jurisdiction boundaries fetched. Cannot perform spatial join.")

filtered_counties = pd.concat(boundary_gdfs, ignore_index=True)
print(f"\nBoundaries fetched: {len(filtered_counties)} jurisdiction(s)")
filtered_counties[['jurisdiction', 'geometry']].head(10)

## Cell 10 — Phase 1c: Spatial Join and County-Level Aggregation

In [ ]:
gdf_gedi = gpd.GeoDataFrame(
    df_gedi,
    geometry=gpd.points_from_xy(df_gedi.longitude, df_gedi.latitude),
    crs="EPSG:4326"
)

print("Performing spatial join...")
gedi_with_county = gpd.sjoin(
    gdf_gedi,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

gedi_county_summary = (
    gedi_with_county
    .groupby(['year', 'jurisdiction'])['rh98']
    .mean()
    .reset_index()
    .rename(columns={'rh98': 'canopy_height_mean_m'})
)

print(f"Spatial join complete. Rows in summary: {len(gedi_county_summary)}")
gedi_county_summary

## Cell 11 — Save County Summary Locally and Upload to S3

In [ ]:
# Save locally
gedi_county_summary.to_csv(OUTPUT_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_COUNTY_CSV)
s3_county_key       = GEDI_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_COUNTY_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll phases completed successfully!")